# 05 — merge EXPLAIN results into the API→SQL matrix


Puts each statement's plan directly under the statement, in the matrix report
that recorded it:

```
**[3]** `grant_records` · SELECT · 0.31 ms

```sql
SELECT ... FROM POLARIS_SCHEMA.GRANT_RECORDS WHERE grantee_catalog_id = ? ...
```
params: `0, 1002, POLARIS`
EXPLAIN (index absent) — Seq Scan on grant_records · est. 1 row(s) · cost 1637.26
```

**Touches no database and no cluster.** It reads two files per case and writes a
third. Safe to run anywhere, any number of times.

## The join

Each explain in the run file carries its own `sql` and `params`, and that pair is
what `query_profile` groups on in the first place — so it is a real key, not an
alignment assumption. Position is NOT usable: `explains` and the matrix's pairs
were zipped at write time and neither list survives re-reading in order.

Measured on the 2026-09-03 admin pair: **561 of 564 statement blocks join**, and
the 3 that do not are the `principal_authentication_data` statements whose
parameters were redacted at capture. Cell 4 asserts that every miss is one of
those — an unexplained miss means the wrong run file was paired with the wrong
report, which is exactly the mistake this notebook could otherwise make silently.

## Re-running

Annotations are REPLACED, never stacked. Running twice gives the same file, so a
corrected run file can be merged over a previous attempt.


In [18]:
# --- cell 1: bootstrap ------------------------------------------------------
import gc
import json
import pathlib
import sys
from datetime import datetime

sys.path.insert(0, str(pathlib.Path.cwd().parent.parent / "src"))

import api_report as rep  # noqa: E402

HERE = pathlib.Path.cwd()
RUNS = HERE / "runs"
REPORTS = HERE / "reports"
STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

#: No init_env, no cluster, no database. This notebook is a file transform --
#: adding a connection would be adding a way for it to fail.
print(f"reports: {REPORTS}")
print(f"runs   : {RUNS}")
print(f"stamp  : {STAMP}")


reports: /Users/kade/hynix/polaris-practice/polaris-learning/diagnostics/api-sql-profile/reports
runs   : /Users/kade/hynix/polaris-practice/polaris-learning/diagnostics/api-sql-profile/runs
stamp  : 20260903-142636


## Inputs — set these by hand

One entry per case: the matrix report to annotate, and the EXPLAIN run whose
plans go into it. `index_state` is the label printed on every EXPLAIN line —
take it from the run's own `index_state` field, which cell 3 checks for you.

Pair them deliberately. A report and a run from different sweeps will still
join on most statements and be wrong about all of them.


In [19]:
# --- cell 2: INPUTS ---------------------------------------------------------
MERGE = {
    "unauthorized": {
        "matrix": "doc-api-sql-matrix-unauthorized-explained-20260903-141115.md",
        "explain": "apiexplain-unauthorized-20260903-142053.json",
    },
    "authorized": {
        "matrix": "doc-api-sql-matrix-authorized-explained-20260903-141115.md",
        "explain": "apiexplain-authorized-20260903-142055.json",
    },
    "admin": {
        "matrix": "doc-api-sql-matrix-admin-explained-20260903-141115.md",
        "explain": "apiexplain-admin-20260903-142056.json",
    },
}

#: Write `doc-api-sql-matrix-<case>-explained-latest.md` as well as the stamped
#: copy. Only `doc-*-latest.md` is tracked by reports/.gitignore.
WRITE_LATEST = True

for case, io in MERGE.items():
    print(f"{case:<13} {io['matrix']}")
    print(f"{'':<13} + {io['explain']}")


unauthorized  doc-api-sql-matrix-unauthorized-explained-20260903-141115.md
              + apiexplain-unauthorized-20260903-142053.json
authorized    doc-api-sql-matrix-authorized-explained-20260903-141115.md
              + apiexplain-authorized-20260903-142055.json
admin         doc-api-sql-matrix-admin-explained-20260903-141115.md
              + apiexplain-admin-20260903-142056.json


In [20]:
# --- cell 3: check the inputs before touching them --------------------------
LOADED = {}
for case, io in MERGE.items():
    m = REPORTS / io["matrix"]
    r = RUNS / io["explain"]
    assert m.exists(), f"{case}: no such report — {m.name}"
    assert r.exists(), f"{case}: no such run — {r.name}"
    d = json.loads(r.read_text(encoding="utf-8"))
    #: The run records the case it drove. Pairing a report with another case's
    #: run is the one mistake that produces a plausible-looking wrong document,
    #: and it is one string comparison to rule out.
    assert d.get("case") == case, (
        f"{case}: {r.name} is a '{d.get('case')}' run. Refusing — a report "
        "annotated with another identity's plans reads as correct and is not."
    )
    assert d.get("matrix", "").endswith(io["matrix"]), (
        f"{case}: {r.name} was taken against {d.get('matrix')}, not "
        f"{io['matrix']}. The plans would describe different statements."
    )
    LOADED[case] = {"text": m.read_text(encoding="utf-8"), "run": d,
                    "matrix_path": m, "run_path": r}
    print(f"{case:<13} index_state={d.get('index_state'):<8} "
          f"pairs={d.get('pairs'):<5} explains={len(d.get('explains') or [])}")
    vol = (d.get("volume") or {}).get("grant_records")
    print(f"{'':<13} grant_records at replay: {vol:,}" if isinstance(vol, int) else "")


unauthorized  index_state=absent   pairs=33    explains=33
              grant_records at replay: 60,815
authorized    index_state=absent   pairs=93    explains=93
              grant_records at replay: 60,815
admin         index_state=absent   pairs=138   explains=138
              grant_records at replay: 60,815


In [21]:
# --- cell 4: merge, and check every miss is one we can name -----------------
MERGED = {}
for case, L in LOADED.items():
    d = L["run"]
    out, stats = rep.annotate_with_explains(
        L["text"], d.get("explains") or [],
        index_state=f"index {d.get('index_state')}",
    )
    MERGED[case] = out
    print(f"{case:<13} matched={stats['matched']:<5} redacted={stats['redacted']:<3} "
          f"unmatched={stats['unmatched']}")
    #: Redacted misses are expected and permanent -- secret-table parameters
    #: are replaced at capture, so those statements can never be replayed.
    #: An UNMATCHED miss is different: it means a statement in the report was
    #: not in the sweep's worklist, which is either a refusal (check the run's
    #: refusal list) or the wrong two files paired together.
    assert stats["unmatched"] == 0, (
        f"{case}: {stats['unmatched']} statement(s) matched no plan and are not "
        "redacted. Either the sweep refused them -- check its dry-run output -- "
        "or this report and this run are from different drives."
    )


unauthorized  matched=390   redacted=0   unmatched=0
authorized    matched=460   redacted=0   unmatched=0
admin         matched=561   redacted=3   unmatched=0


## Spot-check

Read one annotated block before writing anything. The figure to look for is the
grantee lookup on `grant_records`: it is the statement every request issues, and
on the plain upstream schema it plans as a `Seq Scan`.


In [22]:
# --- cell 5: look at it -----------------------------------------------------
import re

_case = "admin" if "admin" in MERGED else next(iter(MERGED))
_txt = MERGED[_case]
_hit = re.search(
    r"\*\*\[\d+\]\*\* `grant_records`[^\n]*\n\n```sql\n.*?\n```\n"
    r"(?:params: `[^`]*`\n)?EXPLAIN[^\n]*\n",
    _txt, re.S,
)
print(f"--- {_case}: first grant_records statement ---\n")
print(_hit.group(0) if _hit else "(no grant_records statement found)")

print("--- EXPLAIN lines by plan, across all cases ---")
_kinds = {}
for case, txt in MERGED.items():
    for line in re.findall(r"^EXPLAIN \([^)]*\) — (.*)$", txt, re.M):
        _kinds[line.split(" · ")[0]] = _kinds.get(line.split(" · ")[0], 0) + 1
for k, n in sorted(_kinds.items(), key=lambda kv: -kv[1])[:12]:
    print(f"  {n:>4}x  {k[:88]}")


--- admin: first grant_records statement ---

(no grant_records statement found)
--- EXPLAIN lines by plan, across all cases ---
   650x  Index Scan on entities using idx_entities
   267x  Index Scan on entities using constraint_name
   173x  Seq Scan on grant_records
    50x  Index Only Scan on grant_records using grant_records_pkey
    44x  Bitmap Heap Scan on entities using idx_entities, entities_pkey
    33x  Bitmap Heap Scan on entities using entities_pkey ×4, idx_entities
    30x  Bitmap Heap Scan on entities using entities_pkey ×5, idx_entities
    19x  ModifyTable, Result
    18x  Bitmap Heap Scan on entities using entities_pkey ×3, idx_entities
    18x  Bitmap Heap Scan on entities using entities_pkey ×4, idx_entities ×2
    16x  Bitmap Heap Scan on entities using entities_pkey ×2, idx_entities
    11x  Bitmap Heap Scan on entities using entities_pkey ×6


In [23]:
# --- cell 6: write ----------------------------------------------------------
REPORTS.mkdir(exist_ok=True)
for case, out in MERGED.items():
    dest = REPORTS / f"doc-api-sql-matrix-{case}-explained-{STAMP}.md"
    dest.write_text(out, encoding="utf-8")
    print(f"-> {dest.name}  ({len(out.splitlines()):,} lines)")
    if WRITE_LATEST:
        latest = REPORTS / f"doc-api-sql-matrix-{case}-explained-latest.md"
        latest.write_text(out, encoding="utf-8")
        print(f"-> {latest.name}")


-> doc-api-sql-matrix-unauthorized-explained-20260903-142636.md  (3,438 lines)
-> doc-api-sql-matrix-unauthorized-explained-latest.md
-> doc-api-sql-matrix-authorized-explained-20260903-142636.md  (4,005 lines)
-> doc-api-sql-matrix-authorized-explained-latest.md
-> doc-api-sql-matrix-admin-explained-20260903-142636.md  (4,848 lines)
-> doc-api-sql-matrix-admin-explained-latest.md


In [24]:
# --- cell 7: teardown -------------------------------------------------------
#: Nothing to dispose -- no connection was opened. The merged texts are large,
#: so they are dropped explicitly per the repo convention.
for _n in ("MERGED", "LOADED", "_txt"):
    if _n in dir():
        del globals()[_n]
gc.collect()
print("done")


done
